#### Física de la Radioterapia. Máster de Física Biomédica. Universidad Complutense de Madrid
# `OpenTPS` en **Colab**
## Cálculo de un tratamiento simple de próstata con haces de **fotones**
-----

Este cuaderno sirve de plantilla de ejemplo de cómo diseñar un tratamiento, optimizarlo y calcular su distribución de dosis asociada mediante `OpenTPS`.


> La instalación en Colab es todavía experimental y requiere utilizar pip sobre la distribución del paquete. No hemos conseguido instalarla utilizando **uv** por problemas de compatibilidad entre las versiones de `numpy` y `scipy`

> El entorno de ejecución puede ser basado en CPU o en GPU. El uso de cupy en `OpenTPS` (entornos GPU`) es experimental. No hemos conseguido las ganancias de velocidad de ejecución de las que habla la documentación de `OpenTPS` que podrían ser hasta de 18X.

In [1]:
import sys

if "google.colab" in sys.modules:
  from IPython import get_ipython
  get_ipython().system('git clone https://gitlab.com/openmcsquare/opentps.git')
  get_ipython().system('pip install ./opentps')
  output = get_ipython().system('nvidia-smi -L')

  # Ensure output_str is always a string, even if output is None
  if output is None:
    output_str = ""
  else:
    output_str = output.decode('utf-8') if isinstance(output, bytes) else output

  if "No devices found" in output_str:
    print("El entorno de ejecución es CPU.")
  else:
    get_ipython().system('uv pip install cupy-cuda12x')


Cloning into 'opentps'...
remote: Enumerating objects: 33170, done.
remote: Counting objects: 100% (4057/4057), done.
remote: Compressing objects: 100% (711/711), done.
remote: Total 33170 (delta 3300), reused 3944 (delta 3237), pack-reused 29113 (from 1)
Receiving objects: 100% (33170/33170), 851.03 MiB | 23.78 MiB/s, done.
Resolving deltas: 100% (24374/24374), done.
Updating files: 100% (1015/1015), done.
Processing ./opentps
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

GPU 0: Tesla T4 (UUID: GPU-9badea75-24d7-43bf-fa1c-22bae3d15f4b)
Using Python 3.12.13 environment at: /usr
Checked 1 package in 215ms


Importación de módulos generales

In [1]:
import os
import logging
import numpy as np
from matplotlib import pyplot as plt
import sys
import copy
from scipy.sparse import csc_matrix

logger = logging.getLogger(__name__)

Importación de módulos de `OpenTPS`

In [2]:
from opentps.core.io.dataLoader import readData
from opentps.core.io.dicomIO import writeRTDose, writeDicomCT, writeRTPlan, writeRTStruct
from opentps.core.processing.planOptimization.tools import evaluateClinical
from opentps.core.data.images import CTImage
from opentps.core.data.images import ROIMask
from opentps.core.data import DVH
from opentps.core.data import Patient
import opentps.core.processing.planOptimization.objectives.dosimetricObjectives as doseObj
from opentps.core.io.scannerReader import readScanner
from opentps.core.io.serializedObjectIO import saveRTPlan, loadRTPlan
from opentps.core.processing.doseCalculation.doseCalculationConfig import DoseCalculationConfig
from opentps.core.processing.imageProcessing.resampler3D import resampleImage3DOnImage3D
from opentps.core.processing.planOptimization.planOptimization import  IntensityModulationOptimizer
from opentps.core.processing.doseCalculation.photons.cccDoseCalculator import CCCDoseCalculator
from opentps.core.data.plan import PhotonPlanDesign

27/04/2026 11:05:56 AM - root - INFO - Loading logging configuration: /usr/local/lib/python3.12/dist-packages/opentps/core/config/logger/logging_config.json
27/04/2026 11:05:56 AM - opentps.core._loggingConfig - INFO - Log level set: INFO


/usr/local/lib/python3.12/dist-packages/cupyx/jit/_interface.py:247: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


27/04/2026 11:06:00 AM - numexpr.utils - INFO - NumExpr defaulting to 2 threads.


## Estudio del paciente y configuración del planificador

Descargar y cargar los datos del paciente de próstata

In [3]:
%%bash
# Ejecutar código Python para descargar los datos del Drive
python << EOF > /dev/null 2>&1
# Descargar los datos compartidos de Google Drive
import gdown
url = 'https://drive.google.com/file/d/1iB2dlt6QKEcI_n92mv9jMbwqy4ZuMKjn/view?usp=sharing'
output = '/content/Prostate.zip'
gdown.download(url, output, quiet=True, fuzzy=True)
EOF

# Extracción de los datos para el cálculo de dosis de ejemplo

# Descomprimir archivo de datos
unzip /content/Prostate.zip -d /content/ >/dev/null

# Eliminar el archivo comprimido
rm -f /content/Prostate.zip

In [4]:
patientDataPath = "/content/PatientID/"
patientData = readData(patientDataPath)

27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 1/127 Dicom files : CT1.2.246.352.221.46289637177583767086002494360155930553.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 2/127 Dicom files : CT1.2.246.352.221.463713292774024375410748421067560572040.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 3/127 Dicom files : CT1.2.246.352.221.464001482127522743612166699020837920151.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 4/127 Dicom files : CT1.2.246.352.221.464209141697279559913540497165389702563.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 5/127 Dicom files : CT1.2.246.352.221.465693264907302273610733635548821593767.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO - Loading data 6/127 Dicom files : CT1.2.246.352.221.468503210913591280717978171650093844911.dcm.
27/04/2026 11:06:38 AM - opentps.core.io.dataLoader - INFO 

In [5]:
# Escáner de simulación
ct = patientData[1]

# Estructuras de interés delimitadas
rt_struct= patientData[0]

#  Listar los nombres de los ROI y los volúmenes objetivo
rt_struct.print_ROINames()



RT Struct UID: 1.2.246.352.205.4680425589062369469.17294819384472381854
  [0]  BODY
  [1]  Vejiga
  [2]  Recto
  [3]  PTV vvss
  [4]  PTV prostata
  [5]  PTV gg
  [6]  Intestino
  [7]  CTV vvss
  [8]  CTV prostata
  [9]  CTV gg
  [10]  Cabeza femoral I
  [11]  Cabeza femoral D
  [12]  Bulbo peneano
  [13]  CouchSurface
  [14]  CouchInterior


Definición del volumen blanco y los órganos de riesgo

In [6]:
# Definir el volumen objetivo
target = rt_struct.getContourByName('PTV prostata').getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)

# Definir los órganos de riesgo relevantes
OAR_rectum = rt_struct.getContourByName("Recto").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)
OAR_bladder = rt_struct.getContourByName("Vejiga").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)
BODY = rt_struct.getContourByName("BODY").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)

## Diseño del plan
-----------
Para mantener simple la plantilla de ejemplo diseñamos un plan con un único campo.

In [7]:
beamNames = ["Beam1"]
gantryAngles = [90.]
couchAngles = [0.]

## Generación cálculo del haz
----

Leer la configuración del escáner

In [ ]:
ctCalibration = readScanner(DoseCalculationConfig().scannerFolder)

In [9]:
# Sobreescribir el comportamiento de stdout y stderr para recibir inmediantamente
# la información de ejecución del cálculo
class FlushingStream:
    def __init__(self, stream):
        self.stream = stream

    def write(self, data):
        self.stream.write(data)
        self.stream.flush()

    def __getattr__(self, attr):
        return getattr(self.stream, attr)

sys.stdout = FlushingStream(sys.stdout)
sys.stderr = FlushingStream(sys.stderr)

In [14]:
## Dose computation from plan
ccc = CCCDoseCalculator(batchSize= 10)
logger.info('Algoritmo de cálculo configurado')
ccc.ctCalibration = readScanner(DoseCalculationConfig().scannerFolder)
logger.info('Configuración Escáner CT leída')

# Load / Generate new plan
output_path = '/content/output'
get_ipython().system('mkdir -p ' + output_path)
plan_file = os.path.join(output_path, "PhotonPlan_WaterPhantom_cropped_resampled.tps")

if os.path.isfile(plan_file): ### Remove the False to load the plan
    plan = loadRTPlan(plan_file, radiationType='photon')
    logger.info('Plan loaded')
    logger.info(f'plan.planDesign.beamlets._doseGrid: {plan.planDesign.beamlets._gridSize}')
else:
    logger.info('El plan no se ha definido previamente. Diseñar el plan...')
    planDesign = PhotonPlanDesign()
    planDesign.ct = ct
    planDesign.targetMask = target
    planDesign.isocenterPosition_mm = None # None take the center of mass of the target
    planDesign.gantryAngles = gantryAngles
    planDesign.couchAngles = couchAngles
    planDesign.beamNames = beamNames
    planDesign.calibration = ctCalibration
    planDesign.xBeamletSpacing_mm = 25
    planDesign.yBeamletSpacing_mm = 25
    planDesign.targetMargin = 2.5
    planDesign.defineTargetMaskAndPrescription(target = target, targetPrescription = 20.)

    plan = planDesign.buildPlan()
    logger.info('Plan diseñado!')

    logger.info('Calcular beamlets...')
    beamlets = ccc.computeBeamlets(ct, plan)
    doseInfluenceMatrix = copy.deepcopy(beamlets)
    logger.info('Beamlets calculados!')

    plan.planDesign.beamlets = beamlets
    logger.info('Beamlets cargados.')
    beamlets.storeOnFS(os.path.join(output_path, "BeamletMatrix_" + plan.seriesInstanceUID + ".blm"))
    logger.info('Beamlets almacenados en el Sistema de archivos')
    # Save plan with initial spot weights in serialized format (OpenTPS format)
    saveRTPlan(plan, plan_file)
    logger.info('Plan creado y guardado en el sistema de archivos.')

27/04/2026 11:16:09 AM - __main__ - INFO - Algoritmo de cálculo configurado
27/04/2026 11:16:09 AM - __main__ - INFO - Configuración Escáner CT leída
27/04/2026 11:16:09 AM - __main__ - INFO - El plan no se ha definido previamente. Diseñar el plan...
27/04/2026 11:16:11 AM - opentps.core.data.plan._photonPlanDesign - INFO - Building plan ...
27/04/2026 11:16:11 AM - opentps.core.processing.planOptimization.planInitializer_Photons - INFO - Target is dilated using a margin of 2.5 mm. This process might take some time.
27/04/2026 11:16:11 AM - opentps.core.processing.imageProcessing.roiMasksProcessing - INFO - Using cupy to dilate mask
27/04/2026 11:16:14 AM - opentps.core.processing.planOptimization.planInitializer_Photons - INFO - Dilation done.
27/04/2026 11:16:14 AM - opentps.core.data.plan._photonPlanDesign - INFO - New photon plan created in 3.916278600692749 sec
27/04/2026 11:16:14 AM - opentps.core.data.plan._photonPlanDesign - INFO - Number of beamlets: 9
27/04/2026 11:16:14 AM -

Definición de objetivos

In [15]:
import opentps.core.processing.planOptimization.objectives.dosimetricObjectives as doseObj
# Objetivos
# ----------

plan.planDesign.objectives.addObjective(doseObj.DMax(target, 70.5, 1.0))
plan.planDesign.objectives.addObjective(doseObj.DMin(target, 69.5, 1.0))
# Other examples of objectives
plan.planDesign.objectives.addObjective(doseObj.DMaxMean(target, 70, 1.0))
plan.planDesign.objectives.addObjective(doseObj.DMinMean(target, 70, 1.0))
# plan.planDesign.objectives.addObjective(roi, doseObj.DUNIFORM, 20, 1.0)
plan.planDesign.objectives.addObjective(doseObj.DVHMin(roi=target, limitValue=69, volume=0.95, weight=1.))
plan.planDesign.objectives.addObjective(doseObj.DVHMax(roi=target, limitValue=71, volume=0.05, weight=1.))
# plan.planDesign.objectives.addObjective(roi, doseObj.EUDMin, 19.5, 1.0, EUDa = 0.2)
# plan.planDesign.objectives.addObjective(roi, doseObj.EUDMax, 20, 1.0, EUDa = 1)
# plan.planDesign.objectives.addObjective(roi, doseObj.EUDUNIFORM, 20.5, 1.0, EUDa = 0.5)
plan.planDesign.objectives.addObjective(doseObj.DFallOff(roi=BODY, target=target, weight=10, fallOffDistance=1, fallOffLowDoseLevel=10, fallOffHighDoseLevel=71))

Optimizar el plan

In [16]:
plan.numberOfFractionsPlanned = 30

plan.planDesign.ROI_cropping = False # False, not cropping allows you to keep the dose outside the ROIs and then use the 'shift' evaluation method, which simply shifts the beamlets.
solver = IntensityModulationOptimizer(method='Scipy_L-BFGS-B', plan=plan, maxiter=1000)

In [17]:
doseImage, ps = solver.optimize()

27/04/2026 11:49:25 AM - opentps.core.processing.planOptimization.planOptimization - INFO - Prepare optimization ...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/opentps/core/processing/planOptimization/objectives/dosimetricObjectives/_DFallOff.py:136: RuntimeWarning: divide by zero encountered in scalar divide
  dfdD = 2/np.sum(self.maskVec) * np.maximum(0, dose[self.maskVec] - self.voxelwiseLimitValue)


27/04/2026 11:49:55 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - Iteration 1 of Scipy-L-BFGS-B
27/04/2026 11:49:56 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - objective = nan  
27/04/2026 11:50:02 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - Iteration 2 of Scipy-L-BFGS-B
27/04/2026 11:50:02 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - objective = nan  
27/04/2026 11:50:09 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - Iteration 3 of Scipy-L-BFGS-B
27/04/2026 11:50:09 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - objective = nan  
27/04/2026 11:50:26 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - Iteration 4 of Scipy-L-BFGS-B
27/04/2026 11:50:27 AM - opentps.core.processing.planOptimization.solvers.scipyOpt - INFO - objective = nan  
27/04/2026 11:50:33 AM - opentps.core.processing.planOptimization.solver

## Resultados

### Exportación de la dosis en formato **DICOM**

> La versión actual tiene un *bug* en el archivo dicomIO.py

Aplicar el parche a `dicomIO.py`...

In [20]:
import os

dicomIO_path = '/usr/local/lib/python3.12/dist-packages/opentps/core/io/dicomIO.py'

# Leer el contenido del archivo
with open(dicomIO_path, 'r') as f:
    lines = f.readlines()

# Modificar la línea 933 (basado en el traceback proporcionado)
# Buscamos la línea que contiene .tostring() y la reemplazamos con .tobytes()
# Asegúrate de que esta es la línea correcta en tu instalación
for i, line in enumerate(lines):
    if "dcm_file.PixelData = (dose.imageArray / dcm_file.DoseGridScaling).astype(np.uint16).transpose(2, 1, 0).tostring()" in line:
        lines[i] = line.replace('.tostring()', '.tobytes()')
        print(f"Parche aplicado en la línea {i+1}: {lines[i].strip()}")
        break

# Escribir el contenido modificado de vuelta al archivo
with open(dicomIO_path, 'w') as f:
    f.writelines(lines)

print("Archivo dicomIO.py parcheado correctamente.")

# Recargar a librería modificadda
import importlib
from opentps.core.io import dicomIO

importlib.reload(dicomIO)
print('Module opentps.core.io.dicomIO reloaded successfully.')

Archivo dicomIO.py parcheado correctamente.
Module opentps.core.io.dicomIO reloaded successfully.


In [24]:
writeRTDose(doseImage, output_path)
logger.info('RT dose file saved')

# Save plan with updated spot weights in serialized format (OpenTPS format)
plan_file_optimized = os.path.join(output_path, "Prostate_plan_resampled_optimized.tps")
saveRTPlan(plan, plan_file_optimized)
logger.info('RT plan file saved')

27/04/2026 12:05:18 PM - pydicom - WARNING - Camel case attribute 'SoftwareVersion' used which is not in the element keyword data dictionary
27/04/2026 12:05:18 PM - pydicom - WARNING - Camel case attribute 'OperatorName' used which is not in the element keyword data dictionary
27/04/2026 12:05:18 PM - pydicom - WARNING - Camel case attribute 'Width' used which is not in the element keyword data dictionary
27/04/2026 12:05:18 PM - pydicom - WARNING - A value of type 'int64' cannot be assigned to a tag with VR US.
27/04/2026 12:05:18 PM - pydicom - WARNING - Camel case attribute 'Height' used which is not in the element keyword data dictionary
27/04/2026 12:05:18 PM - pydicom - WARNING - A value of type 'int64' cannot be assigned to a tag with VR US.
27/04/2026 12:05:18 PM - pydicom - WARNING - Camel case attribute 'ColorType' used which is not in the element keyword data dictionary
27/04/2026 12:05:18 PM - pydicom - WARNING - 'FileDataset.is_little_endian' will be removed in v4.0, set 

AttributeError: 'numpy.ndarray' object has no attribute 'tostring'